# CS-781 Phase 4

In [ ]:
# ---------------- CONFIG ----------------
TOKENIZER_PATH = "/kaggle/input/models/harsvij/cap-tok/pytorch/default/1/legal_bpe_v3.pkl"
CKPT_PATH = None            # None = auto-search /kaggle/input for a fusion*.pt checkpoint

MAX_LEN = 1024              # last 1023 tokens + one end token
TAIL_CHARS = 8000           # only the end of the text is tokenised (about 1500+ tokens), to keep tokenisation fast
MICRO_BATCH = 8
ACCUM_STEPS = 2             # report: batch 8 x 2
EPOCHS = 10                 # report
LR = 5e-5                   # report (peak LR, one value for the whole model)
DROPOUT = 0.1               # report
WEIGHT_DECAY = 0.01         # not in report
WARMUP_FRAC = 0.1           # not in report
GRAD_CLIP = 1.0
THRESH_MODE = "fixed"       # "fixed" = 0.5 (report is silent on thresholds) | "tuned" = sweep on dev
SEED = 42

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
OUT_DIR = "/kaggle/working"
RUN_TAG = "phase4_fusion_full"
UPLOAD_TO_HF = False
TARGET_REPO = "hv-240436/cs781-submissions"

In [ ]:
import os, re, json, glob, time, math, pickle, shutil, zipfile, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("Device:", device, "|", torch.cuda.get_device_name(0) if USE_AMP else "cpu")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    HF_TOKEN = UserSecretsClient().get_secret("CS781_WRITE")
    login(token=HF_TOKEN)
    print("Hugging Face login OK")
except Exception as e:
    print("No HF login (fine if the dataset is public):", type(e).__name__)

## Tokenizer (copied unchanged from the Phase-1 notebook)

In [ ]:
import re, pickle


class LegalBPETokenizer:

    def __init__(self, model_path):

        with open(model_path, "rb") as f:
            model = pickle.load(f)

        self.vocab_size = model["vocab_size"]
        self.merges = model["merges"]                     # [((a, b), new_id), ...]
        self.lowercase = model.get("lowercase", False)
        self.special_tokens = model.get("special_tokens", [])
        self.pattern = re.compile(model["pattern"])

        # pair -> rank (lower rank = learned earlier = merged first)
        self.merge_rank = {
            tuple(pair): rank
            for rank, (pair, _) in enumerate(self.merges)
        }

        # id -> bytes table (O(1) decode)
        self.token_bytes = {i: bytes([i]) for i in range(256)}
        for (l, r), new_id in self.merges:
            self.token_bytes[new_id] = self.token_bytes[l] + self.token_bytes[r]

        base = 256 + len(self.merges)
        self.special_to_id = {t: base + i for i, t in enumerate(self.special_tokens)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.eos_id = self.special_to_id.get("<|endoftext|>")

        self._special_re = (
            re.compile("(" + "|".join(re.escape(t) for t in self.special_tokens) + ")")
            if self.special_tokens else None
        )
        self._cache = {}

    def _encode_piece(self, piece_bytes):

        cached = self._cache.get(piece_bytes)
        if cached is not None:
            return cached

        ids = list(piece_bytes)

        while len(ids) > 1:
            best, best_rank = None, None
            for p in zip(ids, ids[1:]):
                r = self.merge_rank.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = p, r
            if best is None:
                break

            new_id = 256 + best_rank
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == best[0] and ids[i + 1] == best[1]:
                    out.append(new_id)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out

        if len(self._cache) < 500_000:
            self._cache[piece_bytes] = ids
        return ids

    def _encode_ordinary(self, text):

        if self.lowercase:
            text = text.lower()

        out = []
        for piece in self.pattern.findall(text):
            out.extend(self._encode_piece(piece.encode("utf-8")))
        return out

    def encode(self, text, add_eos=False):

        if self._special_re is not None:
            ids = []
            for part in self._special_re.split(text):
                if part in self.special_to_id:
                    ids.append(self.special_to_id[part])
                elif part:
                    ids.extend(self._encode_ordinary(part))
        else:
            ids = self._encode_ordinary(text)

        if add_eos:
            ids.append(self.eos_id)

        return ids

    def decode(self, ids):

        chunks = []
        for token_id in ids:
            if token_id in self.id_to_special:
                chunks.append(self.id_to_special[token_id].encode("utf-8"))
            else:
                chunks.append(self.token_bytes[token_id])

        return b"".join(chunks).decode("utf-8", errors="replace")

tokenizer = LegalBPETokenizer(TOKENIZER_PATH)
EOS_ID = tokenizer.eos_id
print("Vocab size:", tokenizer.vocab_size, "| eos id:", EOS_ID)

## Model code (copied unchanged from the Phase-1 notebook)

In [ ]:
%%writefile model_arch_fusion.py

"""RNN + Transformer fusion causal language model.

Block:  x -> x + Mix(norm1(x)) -> x + MLP(norm2(x))
Mix  =  causal self-attention (RoPE) in PARALLEL with a unidirectional GRU on the
        same normed input, merged by a learned gate: (1-g)*attn + g*rnn.
The GRU branch is present in every `rnn_every`-th block. Output head is tied.
"""

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class FusionConfig:

    vocab_size: int

    # defaults mirror the decoder-only run so the two are easy to compare
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    d_ff: int = 2816

    max_seq_len: int = 1024

    dropout: float = 0.0          # < 1 epoch of data: dropout mostly slows learning

    # ---- fusion-specific fields ----
    rnn_hidden: int = 512         # GRU hidden size (projected back to d_model)
    rnn_every: int = 2            # GRU branch in blocks where i % rnn_every == 0; 0 = no RNN (ablation)
    rnn_fp32: bool = True         # run the recurrent branch in fp32 even under fp16 autocast
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5


class RMSNorm(nn.Module):

    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xf = x.float()
        out = xf * torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (out * self.weight.float()).type_as(x)


def build_rope(head_dim: int, max_len: int, theta: float):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)                 # (L, hd/2)
    emb = torch.cat([freqs, freqs], dim=-1)          # (L, hd)
    return emb.cos(), emb.sin()


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    # x: (B, H, T, hd); cos/sin: (T, hd). Computed in fp32, returned in x.dtype
    xf = x.float()
    return (xf * cos + rotate_half(xf) * sin).type_as(x)


class CausalSelfAttention(nn.Module):

    def __init__(self, cfg: FusionConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        assert self.head_dim % 2 == 0, "head_dim must be even for RoPE"
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = cfg.dropout

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)                                   # each (B, H, T, hd)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).reshape(B, T, C)
        return self.out(y)


class RecurrentBranch(nn.Module):
    """Unidirectional GRU (strictly causal) followed by a projection to d_model."""

    def __init__(self, cfg: FusionConfig):
        super().__init__()
        self.fp32 = cfg.rnn_fp32
        self.gru = nn.GRU(cfg.d_model, cfg.rnn_hidden, num_layers=1,
                          batch_first=True, bidirectional=False)
        self.proj = nn.Linear(cfg.rnn_hidden, cfg.d_model, bias=False)

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        out_dtype = h.dtype
        if self.fp32:
            # keep the sequential recurrence out of fp16
            with torch.autocast(device_type=h.device.type, enabled=False):
                y, _ = self.gru(h.float())
        else:
            y, _ = self.gru(h)
        return self.proj(y.to(out_dtype))


class MLP(nn.Module):

    def __init__(self, cfg: FusionConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.fc2 = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class FusionBlock(nn.Module):

    def __init__(self, cfg: FusionConfig, use_rnn: bool):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = CausalSelfAttention(cfg)
        self.rnn = RecurrentBranch(cfg) if use_rnn else None
        self.gate = nn.Linear(cfg.d_model, cfg.d_model, bias=True) if use_rnn else None
        self.norm2 = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = MLP(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin):
        h = self.norm1(x)
        mixed = self.attn(h, cos, sin)
        if self.rnn is not None:
            r = self.rnn(h)
            g = torch.sigmoid(self.gate(h))
            mixed = (1.0 - g) * mixed + g * r
        x = x + self.drop(mixed)
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x


class FusionLM(nn.Module):
    """Contract used by the rest of the notebook:

    forward(idx: LongTensor (B, T)) -> logits FloatTensor (B, T, vocab_size)
      * strictly causal: logits[:, t] depends only on idx[:, :t+1]
      * works for any 1 <= T <= cfg.max_seq_len
      * exposes self.cfg (needs .max_seq_len)
    """

    def __init__(self, cfg: FusionConfig):

        super().__init__()

        self.cfg = cfg

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([
            FusionBlock(cfg, use_rnn=(cfg.rnn_every > 0 and i % cfg.rnn_every == 0))
            for i in range(cfg.n_layers)
        ])
        self.norm_f = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight             # weight tying

        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_seq_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        # GPT-2 style: scale the projections that write into the residual stream
        res_std = 0.02 / math.sqrt(2 * cfg.n_layers)
        for name, p in self.named_parameters():
            if name.endswith(("attn.out.weight", "mlp.fc2.weight", "rnn.proj.weight")):
                nn.init.normal_(p, mean=0.0, std=res_std)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:

        B, T = idx.shape
        assert 1 <= T <= self.cfg.max_seq_len, f"sequence length {T} outside [1, {self.cfg.max_seq_len}]"

        x = self.drop(self.tok_emb(idx))
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]

        for blk in self.blocks:
            x = blk(x, cos, sin)

        x = self.norm_f(x)
        return self.lm_head(x)

In [ ]:
import importlib, sys
sys.modules.pop("model_arch_fusion", None)
model_arch_fusion = importlib.import_module("model_arch_fusion")
FusionConfig, FusionLM = model_arch_fusion.FusionConfig, model_arch_fusion.FusionLM
print("Fusion model code loaded.")

## Load the Phase-1 checkpoint

In [ ]:
if CKPT_PATH is None:
    cands = [p for p in glob.glob("/kaggle/input/**/*.pt", recursive=True) if "fusion" in os.path.basename(p).lower()]
    assert cands, "No fusion*.pt under /kaggle/input: attach the checkpoint or set CKPT_PATH"
    CKPT_PATH = sorted(cands)[-1]
print("Checkpoint:", CKPT_PATH)

ckpt = torch.load(CKPT_PATH, map_location="cpu")
cfg_dict = dict(ckpt["config"]); cfg_dict["dropout"] = DROPOUT      # Phase 1 used 0.0; fine-tuning uses 0.1 (no weights change)
cfg = FusionConfig(**cfg_dict)
assert cfg.vocab_size == tokenizer.vocab_size, "tokenizer / checkpoint vocab mismatch"
assert MAX_LEN <= cfg.max_seq_len, "MAX_LEN exceeds the model context"
backbone = FusionLM(cfg)
backbone.load_state_dict(ckpt["model_state"])
print(f"Loaded step {ckpt.get('step')} | layers={cfg.n_layers} d_model={cfg.d_model} dropout={cfg.dropout} | "
      f"{sum(p.numel() for p in backbone.parameters())/1e6:.1f}M params")
del ckpt

## Data (last 1024 tokens of each case)

In [ ]:
os.makedirs(f"{OUT_DIR}/data_p4", exist_ok=True)
for f in ["Phase_4_CJPE/Train/train.json", "Phase_4_CJPE/Train/dev.json", "Phase_4_CJPE/Test/test.json"]:
    dest = f"{OUT_DIR}/data_p4/{os.path.basename(f)}"
    if not os.path.exists(dest):
        shutil.copyfile(hf_hub_download(HF_DATASET_REPO, f, repo_type="dataset", token=HF_TOKEN), dest)
load = lambda n: json.load(open(f"{OUT_DIR}/data_p4/{n}.json", encoding="utf-8"))
train_data, dev_data, test_data = load("train"), load("dev"), load("test")
print(f"Train {len(train_data)} | Dev {len(dev_data)} | Test {len(test_data)}")
for name, d in [("train", train_data), ("dev", dev_data)]:
    ones = sum(int(x["label"]) for x in d)
    print(f"  {name}: label 1 = {ones} ({100*ones/len(d):.1f}%), label 0 = {len(d)-ones}")

def prepare(data, with_labels=True):
    out = []
    for d in tqdm(data, desc="tokenising"):
        ids = tokenizer.encode(d["text"][-TAIL_CHARS:])[-(MAX_LEN - 1):] + [EOS_ID]   # keep the END of the case
        s = {"id": str(d["id"]), "ids": ids}
        if with_labels: s["label"] = int(d["label"])
        out.append(s)
    return out

train_s, dev_s = prepare(train_data), prepare(dev_data)
test_s = prepare(test_data, with_labels=False)
print("train length: mean %.0f, share at full 1024: %.0f%%" % (
    np.mean([len(s["ids"]) for s in train_s]), 100 * np.mean([len(s["ids"]) == MAX_LEN for s in train_s])))

class DS(Dataset):
    def __init__(self, s): self.s = s
    def __len__(self): return len(self.s)
    def __getitem__(self, i): return self.s[i]

def collate(batch):
    L = max(len(b["ids"]) for b in batch)
    ids = torch.zeros(len(batch), L, dtype=torch.long)       # right-pad; causal model never reads padding
    for j, b in enumerate(batch): ids[j, :len(b["ids"])] = torch.tensor(b["ids"])
    out = {"ids": ids, "lengths": torch.tensor([len(b["ids"]) for b in batch]), "id": [b["id"] for b in batch]}
    if "label" in batch[0]: out["label"] = torch.tensor([b["label"] for b in batch])
    return out

train_loader = DataLoader(DS(train_s), batch_size=MICRO_BATCH, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(DS(dev_s), batch_size=16, shuffle=False, collate_fn=collate)
test_loader = DataLoader(DS(test_s), batch_size=16, shuffle=False, collate_fn=collate)

## Classifier (full fine-tuning)

In [ ]:
class FusionBinaryClassifier(nn.Module):
    def __init__(self, backbone, dropout=0.1):
        super().__init__()
        self.bb = backbone
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(backbone.cfg.d_model, 2)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)

    def forward(self, ids, lengths):
        B, T = ids.shape
        x = self.bb.tok_emb(ids)
        cos, sin = self.bb.rope_cos[:T], self.bb.rope_sin[:T]
        for blk in self.bb.blocks:
            x = blk(x, cos, sin)
        x = self.bb.norm_f(x)
        pooled = x[torch.arange(B, device=x.device), lengths.to(x.device) - 1]   # final token = end of the ruling
        return self.head(self.drop(pooled)).float()

model = FusionBinaryClassifier(backbone, DROPOUT).to(device)
n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Full fine-tuning: trainable {n:,} parameters (all)")

@torch.no_grad()
def predict_probs(m, loader):
    m.eval(); P, Y, I = [], [], []
    for b in loader:
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            lg = m(b["ids"].to(device), b["lengths"])
        P += F.softmax(lg, -1)[:, 1].cpu().tolist(); I += b["id"]
        if "label" in b: Y += b["label"].tolist()
    return np.array(P), (np.array(Y) if Y else None), I

def macro_f1(y, p): return f1_score(y, p, average="macro", labels=[0, 1], zero_division=0)

## Training (10 epochs, best epoch chosen on dev macro-F1 at threshold 0.5)

In [ ]:
decay = [p for n_, p in model.named_parameters() if p.requires_grad and p.dim() >= 2]
no_decay = [p for n_, p in model.named_parameters() if p.requires_grad and p.dim() < 2]
optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY},
                               {"params": no_decay, "weight_decay": 0.0}], lr=LR)
steps_per_epoch = math.ceil(len(train_loader) / ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS
warm = int(WARMUP_FRAC * total_steps)
lr_scale = lambda s: (s + 1) / max(1, warm) if s < warm else max(0.0, (total_steps - s) / max(1, total_steps - warm))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_scale)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
criterion = nn.CrossEntropyLoss()

best_f1, best_state, log = -1.0, None, []
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); run = 0.0
    optimizer.zero_grad(set_to_none=True)
    for step, b in enumerate(train_loader):
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            lg = model(b["ids"].to(device), b["lengths"])
        loss = criterion(lg, b["label"].to(device)) / ACCUM_STEPS
        scaler.scale(loss).backward()
        run += loss.item() * ACCUM_STEPS
        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            optimizer.zero_grad(set_to_none=True)
    P, Y, _ = predict_probs(model, dev_loader)
    mf1 = macro_f1(Y, (P >= 0.5).astype(int))
    log.append({"epoch": epoch, "train_loss": run / len(train_loader), "dev_macro_f1_at_0.5": float(mf1)})
    print(f"Epoch {epoch}/{EPOCHS} | loss {run/len(train_loader):.4f} | dev macro-F1 @0.5 {mf1:.4f} | {time.time()-t0:.0f}s")
    if mf1 > best_f1:
        best_f1 = mf1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
json.dump(log, open(f"{OUT_DIR}/train_log_{RUN_TAG}.json", "w"))
print(f"Restored best epoch: dev macro-F1 @0.5 = {best_f1:.4f}")

## Dev report and decision threshold
The report does not mention threshold tuning, so the default is a fixed 0.5. Both numbers are printed so you can see what tuning would add. If you switch to `THRESH_MODE = "tuned"`, say so in the report: the threshold is chosen on the same dev split it is scored on, so the result is optimistic.

In [ ]:
P, Y, _ = predict_probs(model, dev_loader)
p05 = (P >= 0.5).astype(int)
print(f"Dev @0.5: macro-F1 {macro_f1(Y, p05):.4f} | P(label 1) {precision_score(Y, p05, zero_division=0):.4f} | "
      f"R(label 1) {recall_score(Y, p05, zero_division=0):.4f}")

best_th, best_tuned = 0.5, macro_f1(Y, p05)
for th in np.arange(0.30, 0.70, 0.005):
    f = macro_f1(Y, (P >= th).astype(int))
    if f > best_tuned: best_tuned, best_th = f, float(th)
print(f"Dev tuned: threshold {best_th:.3f} -> macro-F1 {best_tuned:.4f} (optimistic: tuned on dev)")

THRESHOLD = best_th if THRESH_MODE == "tuned" else 0.5
print("Using threshold:", THRESHOLD, f"(THRESH_MODE={THRESH_MODE})")

## Test predictions and submission zip

In [ ]:
PT, _, test_ids = predict_probs(model, test_loader)
np.save(f"{OUT_DIR}/test_probs_{RUN_TAG}.npy", PT)

final = [{"id": str(i), "pred": int(p >= THRESHOLD)} for i, p in zip(test_ids, PT)]
assert len(final) == len(test_data), f"count mismatch: {len(final)} vs {len(test_data)}"
assert [r["id"] for r in final] == [str(d["id"]) for d in test_data], "id order differs from the test file"
print("share predicted label 1:", round(float(np.mean([r["pred"] for r in final])), 3))

pred_path = f"{OUT_DIR}/prediction.json"
zip_path = f"{OUT_DIR}/submission_{RUN_TAG}.zip"
json.dump(final, open(pred_path, "w", encoding="utf-8"), indent=2)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")
print("Created", zip_path, f"({os.path.getsize(zip_path)/1024:.1f} KB)", zipfile.ZipFile(zip_path).namelist())

if UPLOAD_TO_HF:
    from huggingface_hub import HfApi
    HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=zip_path, path_in_repo=f"phase4/{os.path.basename(zip_path)}",
                                      repo_id=TARGET_REPO, repo_type="dataset")
    print("Uploaded to Hugging Face")